In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
print("Dataset Information:")
df.info()

print("\nDataset Statistics:")
display(df.describe())

print("\nColumn Names:")
print(df.columns.tolist())

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  704

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000



Column Names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [ ]:
missing_values = df.isnull().sum()

print("Missing values in each column:")
print(missing_values)

print("\nTotal missing values:")
print(missing_values.sum())

Missing values in each column:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Total missing values:
0


In [ ]:
# Check for blank or whitespace-only values
blank_values = (df.astype(str).apply(lambda x: x.str.strip() == "")).sum()

print("Blank values in each column:")
print(blank_values)

print("\nTotal blank values:")
print(blank_values.sum())

Blank values in each column:
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

Total blank values:
11


In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing TotalCharges:", df["TotalCharges"].isnull().sum())
print("Data type:", df["TotalCharges"].dtype)

Missing TotalCharges: 11
Data type: float64


In [ ]:
# Fill missing TotalCharges values using the median

median_total_charges = df["TotalCharges"].median()

df["TotalCharges"] = df["TotalCharges"].fillna(median_total_charges)

print("Median TotalCharges:", median_total_charges)
print("Missing TotalCharges after cleaning:",
      df["TotalCharges"].isnull().sum())

Median TotalCharges: 1397.475
Missing TotalCharges after cleaning: 0


In [ ]:
# Check for duplicate rows

duplicate_rows = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_rows)

Number of duplicate rows: 0


In [ ]:
# Check for duplicate customer IDs

duplicate_ids = df["customerID"].duplicated().sum()

print("Number of duplicate customer IDs:", duplicate_ids)

Number of duplicate customer IDs: 0


In [ ]:
# Function to detect outliers using the IQR method

def check_outliers(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    print("Column:", column)
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower bound:", lower_bound)
    print("Upper bound:", upper_bound)
    print("Number of potential outliers:", len(outliers))
    print("-" * 40)


check_outliers("tenure")
check_outliers("MonthlyCharges")
check_outliers("TotalCharges")

Column: tenure
Q1: 9.0
Q3: 55.0
IQR: 46.0
Lower bound: -60.0
Upper bound: 124.0
Number of potential outliers: 0
----------------------------------------
Column: MonthlyCharges
Q1: 35.5
Q3: 89.85
IQR: 54.349999999999994
Lower bound: -46.02499999999999
Upper bound: 171.375
Number of potential outliers: 0
----------------------------------------
Column: TotalCharges
Q1: 402.225
Q3: 3786.6
IQR: 3384.375
Lower bound: -4674.3375
Upper bound: 8863.1625
Number of potential outliers: 0
----------------------------------------


In [ ]:
# Validate important numerical ranges

print("Tenure range:")
print("Minimum:", df["tenure"].min())
print("Maximum:", df["tenure"].max())

print("\nMonthlyCharges range:")
print("Minimum:", df["MonthlyCharges"].min())
print("Maximum:", df["MonthlyCharges"].max())

print("\nTotalCharges range:")
print("Minimum:", df["TotalCharges"].min())
print("Maximum:", df["TotalCharges"].max())

print("\nInvalid tenure values:",
      ((df["tenure"] < 0) | (df["tenure"] > 72)).sum())

print("Negative MonthlyCharges:",
      (df["MonthlyCharges"] < 0).sum())

print("Negative TotalCharges:",
      (df["TotalCharges"] < 0).sum())

Tenure range:
Minimum: 0
Maximum: 72

MonthlyCharges range:
Minimum: 18.25
Maximum: 118.75

TotalCharges range:
Minimum: 18.8
Maximum: 8684.8

Invalid tenure values: 0
Negative MonthlyCharges: 0
Negative TotalCharges: 0


In [ ]:
# Check unique values in important categorical columns

categorical_columns = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "InternetService",
    "Contract",
    "PaymentMethod",
    "Churn"
]

for column in categorical_columns:
    print("\n", column)
    print(df[column].unique())


 gender
['Female' 'Male']

 Partner
['Yes' 'No']

 Dependents
['No' 'Yes']

 PhoneService
['No' 'Yes']

 InternetService
['DSL' 'Fiber optic' 'No']

 Contract
['Month-to-month' 'One year' 'Two year']

 PaymentMethod
['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']

 Churn
['No' 'Yes']


In [ ]:
# Final data quality check

print("Final dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

Final dataset shape: (7043, 21)

Missing values:
0

Duplicate rows:
0

Data types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object


In [ ]:
# Save the cleaned dataset

df.to_csv("Telco_Customer_Churn_Cleaned.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [ ]:
# Final summary of the cleaned dataset

print("FINAL DATA CLEANING SUMMARY")
print("----------------------------")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("TotalCharges data type:", df["TotalCharges"].dtype)
print("MonthlyCharges data type:", df["MonthlyCharges"].dtype)

FINAL DATA CLEANING SUMMARY
----------------------------
Rows: 7043
Columns: 21
Missing values: 0
Duplicate rows: 0
TotalCharges data type: float64
MonthlyCharges data type: float64


In [21]:
# Check for outliers using IQR

def check_outliers(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]

    print(f"\nColumn: {column}")
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower bound:", lower_bound)
    print("Upper bound:", upper_bound)
    print("Number of potential outliers:", len(outliers))

check_outliers("tenure")
check_outliers("MonthlyCharges")
check_outliers("TotalCharges")


Column: tenure
Q1: 9.0
Q3: 55.0
IQR: 46.0
Lower bound: -60.0
Upper bound: 124.0
Number of potential outliers: 0

Column: MonthlyCharges
Q1: 35.5
Q3: 89.85
IQR: 54.349999999999994
Lower bound: -46.02499999999999
Upper bound: 171.375
Number of potential outliers: 0

Column: TotalCharges
Q1: 402.225
Q3: 3786.6
IQR: 3384.375
Lower bound: -4674.3375
Upper bound: 8863.1625
Number of potential outliers: 0


In [22]:
# Check for duplicate rows

duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


In [23]:
# Final data quality check

print("FINAL DATA QUALITY CHECK")
print("-" * 30)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Data types:")
print(df.dtypes)

FINAL DATA QUALITY CHECK
------------------------------
Rows: 7043
Columns: 21
Missing values: 0
Duplicate rows: 0
Data types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object


In [24]:
# Save the final cleaned dataset

df.to_csv("Telco_Customer_Churn_Cleaned.csv", index=False)

print("Final cleaned dataset saved successfully!")

Final cleaned dataset saved successfully!
